# Trading Intelligent — CH03 v2.1 — Notebook 1
## ARCH/GARCH : dépendance de volatilité, estimation et diagnostics

**Auteur académique : Tawfik Masrour**

### Question directrice
Comment une série peut-elle être presque imprévisible dans son **signe** tout en restant fortement prévisible dans son **niveau de risque** ?

### Objectifs
1. visualiser le clustering de volatilité ;
2. comparer l'ACF des rendements et de leurs carrés ;
3. appliquer ARCH-LM ;
4. estimer explicitement un GARCH(1,1) ;
5. interpréter $\omega$, $\alpha$, $\beta$, la persistance et la demi-vie ;
6. diagnostiquer les résidus standardisés.

> Convention : le CSV stocke les rendements en décimal. L'estimation GARCH ci-dessous travaille en pourcentage pour obtenir des nombres plus lisibles.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import norm, t as student_t, chi2
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

SEED = 20261001
DATASET_ID = "TI_CH03_VOL_RISK_V1"
DATASET_FILENAME = "TI_CH03_Vol_Risk_v1.csv"
np.random.seed(SEED)


def generate_dataset(seed=SEED, n=1700):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2020-01-02", periods=n)
    mu = 0.00018
    alpha, beta, gamma = 0.055, 0.87, 0.10
    target_var = 0.00010
    nu = 7
    omega = target_var * (1 - alpha - beta - gamma / 2)
    z = rng.standard_t(df=nu, size=n) / np.sqrt(nu / (nu - 2))
    eps = np.zeros(n)
    sig2 = np.full(n, target_var)
    r = np.zeros(n)
    for i in range(1, n):
        indicator = 1.0 if eps[i - 1] < 0 else 0.0
        sig2[i] = (
            omega
            + alpha * eps[i - 1] ** 2
            + gamma * indicator * eps[i - 1] ** 2
            + beta * sig2[i - 1]
        )
        eps[i] = np.sqrt(sig2[i]) * z[i]
        r[i] = mu + eps[i]
    price = 100 * np.exp(np.cumsum(r))
    volume = np.exp(
        15.7 + 18 * np.sqrt(sig2) + 45 * np.abs(r) + rng.normal(0, 0.25, n)
    ).astype("int64")
    return pd.DataFrame({
        "date": dates,
        "return": r,
        "price": price,
        "true_sigma": np.sqrt(sig2),
        "innovation": eps,
        "volume": volume,
    })


def locate_dataset(filename=DATASET_FILENAME):
    here = Path.cwd()
    roots = [here, here.parent, here.parent.parent]
    subdirs = [Path('.'), Path('datasets'), Path('data'), Path('resources')]
    for root in roots:
        for sub in subdirs:
            candidate = (root / sub / filename).resolve()
            if candidate.exists():
                return candidate
    return None


path = locate_dataset()
if path is not None:
    df = pd.read_csv(path, parse_dates=['date'])
    print(f"{DATASET_ID} chargé depuis : {path}")
else:
    df = generate_dataset()
    print(f"{DATASET_ID} absent localement : version déterministe régénérée en mémoire.")
print("Dimensions :", df.shape)
print("Convention : return et true_sigma sont stockés en décimal.")


def garch_variance(residuals, omega, alpha, beta):
    e = np.asarray(residuals, float)
    v = np.empty_like(e)
    v[0] = max(np.var(e), 1e-8)
    for i in range(1, len(e)):
        v[i] = omega + alpha * e[i - 1] ** 2 + beta * v[i - 1]
    return v


def fit_garch11(r_dec):
    x = np.asarray(r_dec, float) * 100.0  # estimation en pourcentage
    mu = x.mean()
    e = x - mu
    sample_var = max(np.var(e), 1e-6)

    def objective(p):
        omega, alpha, beta = p
        if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 0.999:
            return 1e12
        v = garch_variance(e, omega, alpha, beta)
        return 0.5 * np.sum(np.log(2 * np.pi) + np.log(v) + e ** 2 / v)

    result = minimize(
        objective,
        [0.03 * sample_var, 0.08, 0.88],
        method='SLSQP',
        bounds=[(1e-10, None), (0, 0.999), (0, 0.999)],
        constraints=[{'type': 'ineq', 'fun': lambda p: 0.999 - p[1] - p[2]}],
        options={'maxiter': 1200, 'ftol': 1e-10},
    )
    omega, alpha, beta = result.x
    v = garch_variance(e, omega, alpha, beta)
    return {
        'success': bool(result.success),
        'message': str(result.message),
        'mu': mu,
        'omega': omega,
        'alpha': alpha,
        'beta': beta,
        'variance': v,
        'nll': float(result.fun),
        'nobs': len(x),
    }


def gjr_variance(residuals, omega, alpha, gamma, beta):
    e = np.asarray(residuals, float)
    v = np.empty_like(e)
    v[0] = max(np.var(e), 1e-8)
    for i in range(1, len(e)):
        prev = e[i - 1]
        v[i] = (
            omega
            + alpha * prev ** 2
            + gamma * (prev < 0) * prev ** 2
            + beta * v[i - 1]
        )
    return v


def fit_gjr11(r_dec):
    x = np.asarray(r_dec, float) * 100.0
    mu = x.mean()
    e = x - mu
    sample_var = max(np.var(e), 1e-6)

    def objective(p):
        omega, alpha, gamma, beta = p
        if min(omega, alpha, gamma, beta) < 0:
            return 1e12
        if alpha + beta + gamma / 2 >= 0.999:
            return 1e12
        v = gjr_variance(e, omega, alpha, gamma, beta)
        return 0.5 * np.sum(np.log(2 * np.pi) + np.log(v) + e ** 2 / v)

    starts = [
        [0.025 * sample_var, 0.05, 0.10, 0.85],
        [0.030 * sample_var, 0.08, 0.08, 0.82],
        [0.020 * sample_var, 0.04, 0.12, 0.86],
    ]
    candidates = [
        minimize(
            objective,
            start,
            method='Nelder-Mead',
            options={'maxiter': 8000, 'xatol': 1e-10, 'fatol': 1e-8},
        )
        for start in starts
    ]
    result = min(candidates, key=lambda res: res.fun)
    omega, alpha, gamma, beta = result.x
    v = gjr_variance(e, omega, alpha, gamma, beta)
    return {
        'success': bool(result.success),
        'message': str(result.message),
        'mu': mu,
        'omega': omega,
        'alpha': alpha,
        'gamma': gamma,
        'beta': beta,
        'variance': v,
        'nll': float(result.fun),
        'nobs': len(x),
    }


def fit_standardized_t_nu(z):
    z = np.asarray(z, float)

    def nll(nu):
        if nu <= 2.05:
            return 1e12
        scale = np.sqrt(nu / (nu - 2))
        return -np.sum(np.log(scale) + student_t.logpdf(scale * z, df=nu))

    result = minimize_scalar(nll, bounds=(2.1, 80), method='bounded')
    return float(result.x)


def forecast_gjr_one_step(test_returns_dec, fit, last_train_return_dec):
    """Prévisions one-step-ahead avec paramètres figés et mise à jour par le passé observé."""
    test_x = np.asarray(test_returns_dec, float) * 100.0
    mu = fit['mu']
    omega, alpha, gamma, beta = fit['omega'], fit['alpha'], fit['gamma'], fit['beta']
    prev_v = float(fit['variance'][-1])
    prev_e = float(last_train_return_dec * 100.0 - mu)
    out = np.empty(len(test_x))
    for i, x in enumerate(test_x):
        out[i] = omega + alpha * prev_e ** 2 + gamma * (prev_e < 0) * prev_e ** 2 + beta * prev_v
        prev_e = x - mu
        prev_v = out[i]
    return out


def safe_xlogy(count, probability):
    if count == 0:
        return 0.0
    if probability <= 0:
        return -np.inf
    if probability >= 1:
        return 0.0 if np.isclose(probability, 1.0) else -np.inf
    return count * np.log(probability)


def kupiec_test(breaches, alpha_tail):
    hits = np.asarray(breaches, dtype=int)
    n = len(hits)
    x = int(hits.sum())
    p_hat = x / n
    ll0 = safe_xlogy(x, alpha_tail) + safe_xlogy(n - x, 1 - alpha_tail)
    if p_hat in (0, 1):
        ll1 = 0.0
    else:
        ll1 = safe_xlogy(x, p_hat) + safe_xlogy(n - x, 1 - p_hat)
    lr = -2 * (ll0 - ll1)
    return float(lr), float(1 - chi2.cdf(lr, 1))


def christoffersen_independence_test(breaches):
    h = np.asarray(breaches, dtype=int)
    if len(h) < 2:
        return np.nan, np.nan
    n00 = int(np.sum((h[:-1] == 0) & (h[1:] == 0)))
    n01 = int(np.sum((h[:-1] == 0) & (h[1:] == 1)))
    n10 = int(np.sum((h[:-1] == 1) & (h[1:] == 0)))
    n11 = int(np.sum((h[:-1] == 1) & (h[1:] == 1)))
    total = n00 + n01 + n10 + n11
    p = (n01 + n11) / total if total else 0.0
    p01 = n01 / (n00 + n01) if (n00 + n01) else 0.0
    p11 = n11 / (n10 + n11) if (n10 + n11) else 0.0
    ll0 = safe_xlogy(n01 + n11, p) + safe_xlogy(n00 + n10, 1 - p)
    ll1 = (
        safe_xlogy(n01, p01) + safe_xlogy(n00, 1 - p01)
        + safe_xlogy(n11, p11) + safe_xlogy(n10, 1 - p11)
    )
    if not np.isfinite(ll0) or not np.isfinite(ll1):
        return np.nan, np.nan
    lr = -2 * (ll0 - ll1)
    return float(lr), float(1 - chi2.cdf(lr, 1))


def qlike(realized_variance, forecast_variance):
    x = np.asarray(realized_variance, float)
    v = np.asarray(forecast_variance, float)
    ratio = np.maximum(x, 1e-12) / np.maximum(v, 1e-12)
    return ratio - np.log(ratio) - 1


## 1. Observer avant de modéliser

Le **volatility clustering** désigne le regroupement des périodes calmes et agitées. Il ne signifie pas que le prochain signe est facilement prévisible : c'est surtout l'amplitude qui conserve une mémoire.


In [ ]:
r = df.set_index("date")["return"]
fig, axs = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axs[0].plot(r.index, 100 * r, lw=0.65)
axs[0].axhline(0, color="black", lw=0.7)
axs[0].set_ylabel("Rendement (%)")
axs[0].set_title("Rendements : le signe reste difficile à prévoir")
axs[0].grid(alpha=0.2)

axs[1].plot(r.index, 100 * r.rolling(20).std(), label="Écart-type glissant 20 jours")
axs[1].plot(df.date, 100 * df.true_sigma, label="Volatilité vraie — oracle pédagogique", alpha=0.8)
axs[1].set_ylabel("Volatilité (%)")
axs[1].set_title("Amplitude : périodes calmes et agitées se regroupent")
axs[1].legend()
axs[1].grid(alpha=0.2)
plt.show()


## 2. ACF : mémoire de la moyenne ou mémoire de l'amplitude ?

Si $r_t$ possède une faible autocorrélation mais que $r_t^2$ est autocorrélé, la moyenne conditionnelle peut rester simple tandis que la variance conditionnelle est structurée.


In [ ]:
def stem_acf(x, title, nlags=30):
    values = acf(pd.Series(x).dropna(), nlags=nlags, fft=True)
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.stem(range(len(values)), values, basefmt=" ")
    ax.set(title=title, xlabel="Retard", ylabel="ACF")
    ax.grid(alpha=0.2)
    plt.show()

stem_acf(r, "ACF des rendements")
stem_acf(r ** 2, "ACF des rendements au carré")


## 3. Test ARCH-LM

Le test ARCH-LM recherche une dépendance dans les carrés des résidus. Son hypothèse nulle est l'absence d'effets ARCH jusqu'à l'ordre choisi.

Une petite p-value ne signifie pas « le GARCH est forcément correct » : elle indique seulement qu'une variance constante est difficile à défendre.


In [ ]:
lm_stat, lm_p, f_stat, f_p = het_arch((100 * r).dropna(), nlags=12)
print(f"ARCH-LM : statistique={lm_stat:.2f}, p-value={lm_p:.4g}")


## 4. Estimer un GARCH(1,1)

Le modèle est
$$
\sigma_t^2=\omega+\alpha\varepsilon_{t-1}^2+\beta\sigma_{t-1}^2.
$$

- $\omega$ : niveau de base ;
- $\alpha$ : réaction immédiate au dernier choc ;
- $\beta$ : mémoire de la variance passée.

L'estimation par maximum de vraisemblance est écrite explicitement avec SciPy : aucune « boîte noire » spécialisée n'est nécessaire.


In [ ]:
fit = fit_garch11(r)
summary = {
    "convergence": fit["success"],
    "message": fit["message"],
    "mu (%)": fit["mu"],
    "omega": fit["omega"],
    "alpha": fit["alpha"],
    "beta": fit["beta"],
    "alpha+beta": fit["alpha"] + fit["beta"],
}
summary


In [ ]:
sigma_hat = np.sqrt(fit["variance"])
persistence = fit["alpha"] + fit["beta"]
long_var = fit["omega"] / (1 - persistence)
half_life = np.log(0.5) / np.log(persistence)

print(f"Volatilité de long terme : {np.sqrt(long_var):.3f} %")
print(f"Demi-vie d'un choc       : {half_life:.2f} jours")

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(df.date, 100 * df.true_sigma, label="Volatilité vraie — oracle", lw=1.2)
ax.plot(df.date, sigma_hat, label="GARCH estimé (%)", lw=1.0, alpha=0.85)
ax.set_ylabel("Volatilité quotidienne (%)")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


### Calcul à la main : une prévision à un pas

Avec $\omega=0.04$, $\alpha=0.08$, $\beta=0.88$, $\varepsilon_t=-1.5$ et $\sigma_t^2=1.20$ en unités de pourcentage :
$$
\sigma_{t+1}^2=0.04+0.08(1.5)^2+0.88(1.20)=1.276,
$$
donc $\sigma_{t+1}\approx1.13\%$.

Cette décomposition est le cœur du mécanisme GARCH.


## 5. Diagnostiquer les résidus standardisés

Après estimation, on construit
$$
z_t=\frac{\varepsilon_t}{\hat\sigma_t}.
$$

Un bon modèle de variance doit notamment réduire la dépendance présente dans les carrés des résidus.


In [ ]:
e = 100 * r.to_numpy() - fit["mu"]
z = e / sigma_hat

print("Ljung-Box sur z :")
display(acorr_ljungbox(z, lags=[10, 20], return_df=True))
print("Ljung-Box sur z² :")
display(acorr_ljungbox(z ** 2, lags=[10, 20], return_df=True))

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(df.date, z, lw=0.6)
axs[0].set_title("Résidus standardisés")
axs[0].grid(alpha=0.2)
values = acf(z ** 2, nlags=25, fft=True)
axs[1].stem(range(len(values)), values, basefmt=" ")
axs[1].set_title("ACF des carrés standardisés")
axs[1].grid(alpha=0.2)
plt.show()


## 6. Mini-défi étudiant

1. Estimez le GARCH sur les 900 premières observations puis sur les 900 dernières. Comparez la persistance.
2. Calculez la variance de long terme $\omega/(1-\alpha-\beta)$ lorsque la condition est satisfaite.
3. Calculez la demi-vie $\log(0.5)/\log(\alpha+\beta)$.
4. Expliquez pourquoi une ACF faible des rendements n'empêche pas une prévision utile de la volatilité.
5. Donnez une raison pour laquelle une persistance proche de 1 peut être trompeuse.


In [ ]:
# Votre travail commence ici.
# Q1

# Q2

# Q3

# Q4 et Q5 : réponses argumentées dans une cellule Markdown.


### À retenir

Un GARCH ne prétend pas nécessairement prévoir le **signe** du rendement. Il modélise une autre information : l'amplitude de fluctuation plausible compte tenu du passé. Une estimation n'est crédible qu'après contrôle des unités, des contraintes, de la convergence et des résidus standardisés.
